In [2]:
### 재시작 셀 ###
# === 시연 영상 제작 노트북 (분리 버전, 재시작 시 매번 실행) ===
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import importlib
import cv2
import numpy as np
from collections import defaultdict

# 경로 설정
BASE = "/content/drive/MyDrive/driving2"        # 작업 데이터는 그대로
DEMO_BASE = "/content/drive/MyDrive/driving2_demo"  # 시연 영상은 분리

MODULE_DIR = f"{BASE}/modules"

# C 모듈 경로 추가
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

importlib.invalidate_caches()
from behavior_rules import detect_suspects, DEFAULT_THRESHOLDS

# 시연 영상 출력 폴더 생성
os.makedirs(DEMO_BASE, exist_ok=True)
DEMO_OUTPUT = f"{DEMO_BASE}/demo_videos"
os.makedirs(DEMO_OUTPUT, exist_ok=True)

os.chdir(BASE)

print(f"✅ 환경 세팅 완료")
print(f"   작업 데이터: {BASE}")
print(f"   시연 영상 출력: {DEMO_OUTPUT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경 세팅 완료
   작업 데이터: /content/drive/MyDrive/driving2
   시연 영상 출력: /content/drive/MyDrive/driving2_demo/demo_videos


In [3]:
### 시각화 함수 ###
def create_demo_video(video_id, base_path, thresholds=None, max_frames=None):
    """
    시연 영상 생성 — 의심 차량 빨간 박스 + 자막
    """
    print(f"\n{'='*60}")
    print(f"=== {video_id} 시연 영상 생성 ===")
    print(f"{'='*60}")

    json_path = f"{base_path}/outputs/test_tracks_v1.1.json"

    # 원본 영상 경로
    original_video = f"{BASE}/data/videos/{video_id}.mp4"
    if not os.path.exists(original_video):
        original_video = f"{BASE}/data/carla/{video_id}_video.mp4"

    print(f"  JSON: {json_path}")
    print(f"  원본 영상: {original_video}")

    if not os.path.exists(original_video):
        print(f"  ❌ 원본 영상 없음")
        return None

    with open(json_path) as f:
        data = json.load(f)

    # C 룰 적용
    if thresholds:
        result = detect_suspects(data, thresholds=thresholds)
    else:
        result = detect_suspects(data)

    suspect_ids = set(result['suspect_ids'])
    reasons_per_id = {s['track_id']: s['reasons'] for s in result['suspects']}

    print(f"  의심 차량: {len(suspect_ids)}대 (총 {result['total_analyzed']}대 중)")
    print(f"  사유: {result['summary']}")

    # 영상 열기
    cap = cv2.VideoCapture(original_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if max_frames:
        total_frames = min(total_frames, max_frames)

    print(f"  영상: {width}x{height}, {fps:.1f}fps, {total_frames}프레임")

    # 출력 영상 — DEMO_OUTPUT으로
    output_path = f"{DEMO_OUTPUT}/{video_id}_demo.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frames_dict = {f['frame_id']: f for f in data['frames']}

    frame_idx = 0
    while frame_idx < total_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in frames_dict:
            frame_data = frames_dict[frame_idx]

            for v in frame_data['vehicles']:
                tid = v['track_id']
                x1, y1, x2, y2 = [int(c) for c in v['bbox_pixel']]

                if tid in suspect_ids:
                    color = (0, 0, 255)  # 빨간색
                    thickness = 4
                else:
                    color = (0, 255, 0)  # 초록색
                    thickness = 2

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

                label = f"ID {tid}"
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                if tid in suspect_ids:
                    reasons = reasons_per_id.get(tid, [])
                    reason_en = ", ".join(reasons)
                    cv2.putText(frame, reason_en, (x1, y2 + 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # 화면 상단 정보
        info_text = f"{video_id.upper()} | Frame {frame_idx}/{total_frames} | Suspect: {len(suspect_ids)}"
        cv2.rectangle(frame, (0, 0), (width, 40), (0, 0, 0), -1)
        cv2.putText(frame, info_text, (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        out.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f"    {frame_idx}/{total_frames} 처리 중...")

    cap.release()
    out.release()

    file_size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f"  ✅ 완료: {output_path}")
    print(f"     크기: {file_size_mb:.1f}MB")

    return output_path

print("✅ 시각화 함수 정의 완료")

✅ 시각화 함수 정의 완료


In [ ]:
### v01 시연 영상 생성 ###
# v01 (흐름 영상, 시내 교차로)
v01_thresholds = {'tail_gap': 3.0, 'lane_change': 2}

output_path = create_demo_video(
    video_id='v01',
    base_path=f"{BASE}/v01",
    thresholds=v01_thresholds
)


=== v01 시연 영상 생성 ===
  JSON: /content/drive/MyDrive/driving2/v01/outputs/test_tracks_v1.1.json
  원본 영상: /content/drive/MyDrive/driving2/data/videos/v01.mp4
  의심 차량: 46대 (총 57대 중)
  사유: {'lane_change': 22, 'tailgating': 34, 'lane_weaving': 6}
  영상: 1280x720, 24.0fps, 538프레임
    100/538 처리 중...
    200/538 처리 중...
    300/538 처리 중...
    400/538 처리 중...
    500/538 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/v01_demo.mp4
     크기: 17.1MB


In [ ]:
###CCTV 영상 5개에 극엄격 C 룰 적용하여 시각화 ###
# === CCTV 5개 시연 영상 (극엄격 임계값) ===
import os
import sys
import json
import cv2
import numpy as np
import importlib

BASE = "/content/drive/MyDrive/driving2"
DEMO_BASE = "/content/drive/MyDrive/driving2_demo"
MODULE_DIR = f"{BASE}/modules"

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)
importlib.invalidate_caches()
from behavior_rules import detect_suspects, DEFAULT_THRESHOLDS

DEMO_OUTPUT = f"{DEMO_BASE}/demo_videos"
os.makedirs(DEMO_OUTPUT, exist_ok=True)

# 극엄격 임계값
TIGHT_TH = {
    'tail_gap': 1.5,
    'lane_change': 5,
    'weaving_std': 0.50,
    'min_track_frames': 60,
}

def create_demo_video_cctv(video_id, max_frames=None):
    """CCTV 영상 시연 영상 생성 (극엄격 임계값)"""
    print(f"\n{'='*60}")
    print(f"=== {video_id} 시연 영상 (극엄격) ===")
    print(f"{'='*60}")

    json_path = f"{BASE}/{video_id}/outputs/test_tracks_v1.1.json"
    original_video = f"{BASE}/data/videos/{video_id}.mp4"

    if not os.path.exists(original_video):
        print(f"  ❌ 원본 영상 없음: {original_video}")
        return None

    with open(json_path) as f:
        data = json.load(f)

    # C 룰 적용 (극엄격)
    result = detect_suspects(data, thresholds=TIGHT_TH)

    suspect_ids = set(result['suspect_ids'])
    reasons_per_id = {s['track_id']: s['reasons'] for s in result['suspects']}

    print(f"  의심 차량: {len(suspect_ids)}대 (총 {result['total_analyzed']}대 중)")
    print(f"  사유: {result['summary']}")

    # 영상 열기
    cap = cv2.VideoCapture(original_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if max_frames:
        total_frames = min(total_frames, max_frames)

    print(f"  영상: {width}x{height}, {fps:.1f}fps, {total_frames}프레임")

    # 출력 영상
    output_path = f"{DEMO_OUTPUT}/{video_id}_demo_tight.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frames_dict = {f['frame_id']: f for f in data['frames']}

    frame_idx = 0
    while frame_idx < total_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in frames_dict:
            frame_data = frames_dict[frame_idx]

            for v in frame_data['vehicles']:
                tid = v['track_id']
                x1, y1, x2, y2 = [int(c) for c in v['bbox_pixel']]

                if tid in suspect_ids:
                    color = (0, 0, 255)  # 빨간색
                    thickness = 4
                else:
                    color = (0, 255, 0)  # 초록색
                    thickness = 2

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

                label = f"ID {tid}"
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                if tid in suspect_ids:
                    reasons = reasons_per_id.get(tid, [])
                    reason_en = ", ".join(reasons)
                    cv2.putText(frame, reason_en, (x1, y2 + 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        # 화면 상단 정보
        info_text = f"{video_id.upper()} (TIGHT) | Frame {frame_idx}/{total_frames} | Suspect: {len(suspect_ids)}"
        cv2.rectangle(frame, (0, 0), (width, 50), (0, 0, 0), -1)
        cv2.putText(frame, info_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        out.write(frame)
        frame_idx += 1

        if frame_idx % 200 == 0:
            print(f"    {frame_idx}/{total_frames} 처리 중...")

    cap.release()
    out.release()

    file_size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f"  ✅ 완료: {output_path}")
    print(f"     크기: {file_size_mb:.1f}MB")

    return output_path


# === 5개 영상 자동 처리 ===
CCTV_VIDEOS = ['v01', 'v02', 'v04', 'v05', 'v06']

print(f"\n{'#'*60}")
print(f"# CCTV 5개 시연 영상 생성 (극엄격 임계값)")
print(f"# tail_gap=1.5, lane_change=5, min_track=60")
print(f"{'#'*60}")

for video_id in CCTV_VIDEOS:
    create_demo_video_cctv(video_id)

print(f"\n\n{'#'*60}")
print(f"# 5개 시연 영상 생성 완료!")
print(f"{'#'*60}")
print(f"\n저장 위치: {DEMO_OUTPUT}/")
for v in CCTV_VIDEOS:
    output_file = f"{DEMO_OUTPUT}/{v}_demo_tight.mp4"
    if os.path.exists(output_file):
        size_mb = os.path.getsize(output_file) / 1024 / 1024
        print(f"  ✅ {v}_demo_tight.mp4 ({size_mb:.1f}MB)")


############################################################
# CCTV 5개 시연 영상 생성 (극엄격 임계값)
# tail_gap=1.5, lane_change=5, min_track=60
############################################################

=== v01 시연 영상 (극엄격) ===
  의심 차량: 18대 (총 32대 중)
  사유: {'lane_change': 7, 'lane_weaving': 2, 'tailgating': 14}
  영상: 1280x720, 24.0fps, 538프레임
    200/538 처리 중...
    400/538 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/v01_demo_tight.mp4
     크기: 16.3MB

=== v02 시연 영상 (극엄격) ===
  의심 차량: 3대 (총 33대 중)
  사유: {'tailgating': 3, 'lane_change': 1}
  영상: 1280x720, 24.0fps, 608프레임
    200/608 처리 중...
    400/608 처리 중...
    600/608 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/v02_demo_tight.mp4
     크기: 13.9MB

=== v04 시연 영상 (극엄격) ===
  의심 차량: 31대 (총 36대 중)
  사유: {'tailgating': 30, 'lane_weaving': 1}
  영상: 1280x720, 24.0fps, 671프레임
    200/671 처리 중...
    400/671 처리 중...
    600/671 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/v04_demo_tight.m

In [6]:
# === CARLA 5개 시연 영상 자동 생성 ===
import os
import sys
import json
import cv2
import numpy as np
import importlib

BASE = "/content/drive/MyDrive/driving2"
DEMO_BASE = "/content/drive/MyDrive/driving2_demo"
MODULE_DIR = f"{BASE}/modules"

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)
importlib.invalidate_caches()
from behavior_rules import detect_suspects, DEFAULT_THRESHOLDS

DEMO_OUTPUT = f"{DEMO_BASE}/demo_videos"
os.makedirs(DEMO_OUTPUT, exist_ok=True)


def create_demo_video_carla(scenario):
    """CARLA 영상 시연 영상 생성 (기본 임계값)"""
    print(f"\n{'='*60}")
    print(f"=== carla_{scenario} 시연 영상 생성 ===")
    print(f"{'='*60}")

    json_path = f"{BASE}/carla_{scenario}/outputs/test_tracks_v1.1.json"
    original_video = f"{BASE}/data/carla/{scenario}_video.mp4"

    if not os.path.exists(original_video):
        print(f"  ❌ 원본 영상 없음: {original_video}")
        return None

    with open(json_path) as f:
        data = json.load(f)

    # C 룰 적용 (CARLA는 기본 임계값)
    result = detect_suspects(data)

    suspect_ids = set(result['suspect_ids'])
    reasons_per_id = {s['track_id']: s['reasons'] for s in result['suspects']}

    print(f"  의심 차량: {len(suspect_ids)}대 (총 {result['total_analyzed']}대 중)")
    print(f"  사유: {result['summary']}")

    # 영상 열기
    cap = cv2.VideoCapture(original_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"  영상: {width}x{height}, {fps:.1f}fps, {total_frames}프레임")

    output_path = f"{DEMO_OUTPUT}/carla_{scenario}_demo.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frames_dict = {f['frame_id']: f for f in data['frames']}

    scenario_desc = {
        'normal': 'Normal Driving (Control)',
        'lane_weaving': 'Lane Weaving',
        'tailgating': 'Tailgating',
        'sudden_lane_change': 'Sudden Lane Change',
        'speeding': 'Speeding (Rule Inactive)'
    }

    frame_idx = 0
    while frame_idx < total_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in frames_dict:
            frame_data = frames_dict[frame_idx]

            for v in frame_data['vehicles']:
                tid = v['track_id']
                x1, y1, x2, y2 = [int(c) for c in v['bbox_pixel']]

                if tid in suspect_ids:
                    color = (0, 0, 255)
                    thickness = 4
                else:
                    color = (0, 255, 0)
                    thickness = 2

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

                label = f"ID {tid}"
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                if tid in suspect_ids:
                    reasons = reasons_per_id.get(tid, [])
                    reason_en = ", ".join(reasons)
                    cv2.putText(frame, reason_en, (x1, y2 + 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        info_text = f"CARLA | {scenario_desc.get(scenario, scenario)} | Frame {frame_idx}/{total_frames} | Suspect: {len(suspect_ids)}"
        cv2.rectangle(frame, (0, 0), (width, 50), (0, 0, 0), -1)
        cv2.putText(frame, info_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        out.write(frame)
        frame_idx += 1

        if frame_idx % 200 == 0:
            print(f"    {frame_idx}/{total_frames} 처리 중...")

    cap.release()
    out.release()

    file_size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f"  ✅ 완료: {output_path}")
    print(f"     크기: {file_size_mb:.1f}MB")

    return output_path


# === 5개 CARLA 영상 자동 처리 ===
CARLA_SCENARIOS = ['normal', 'lane_weaving', 'tailgating', 'sudden_lane_change', 'speeding']

print(f"\n{'#'*60}")
print(f"# CARLA 5개 시연 영상 생성 시작")
print(f"{'#'*60}")

for scenario in CARLA_SCENARIOS:
    create_demo_video_carla(scenario)

print(f"\n\n{'#'*60}")
print(f"# 5개 시연 영상 생성 완료!")
print(f"{'#'*60}")
print(f"\n저장 위치: {DEMO_OUTPUT}/")
for s in CARLA_SCENARIOS:
    output_file = f"{DEMO_OUTPUT}/carla_{s}_demo.mp4"
    if os.path.exists(output_file):
        size_mb = os.path.getsize(output_file) / 1024 / 1024
        print(f"  ✅ carla_{s}_demo.mp4 ({size_mb:.1f}MB)")


############################################################
# CARLA 5개 시연 영상 생성 시작
############################################################

=== carla_normal 시연 영상 생성 ===
  의심 차량: 0대 (총 5대 중)
  사유: {}
  영상: 1920x1080, 30.0fps, 1050프레임
    200/1050 처리 중...
    400/1050 처리 중...
    600/1050 처리 중...
    800/1050 처리 중...
    1000/1050 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/carla_normal_demo.mp4
     크기: 46.1MB

=== carla_lane_weaving 시연 영상 생성 ===
  의심 차량: 1대 (총 5대 중)
  사유: {'lane_weaving': 1}
  영상: 1920x1080, 30.0fps, 1050프레임
    200/1050 처리 중...
    400/1050 처리 중...
    600/1050 처리 중...
    800/1050 처리 중...
    1000/1050 처리 중...
  ✅ 완료: /content/drive/MyDrive/driving2_demo/demo_videos/carla_lane_weaving_demo.mp4
     크기: 46.0MB

=== carla_tailgating 시연 영상 생성 ===
  의심 차량: 1대 (총 5대 중)
  사유: {'tailgating': 1}
  영상: 1920x1080, 30.0fps, 1050프레임
    200/1050 처리 중...
    400/1050 처리 중...
    600/1050 처리 중...
    800/1050 처리 중...
    1000/1050 처리 중...
  ✅ 완료: /cont

In [4]:
# === Week 4 데모 영상 분석 보고서 ===
import os
import sys
import json
import importlib
from datetime import datetime

BASE = "/content/drive/MyDrive/driving2"
DEMO_BASE = "/content/drive/MyDrive/driving2_demo"
MODULE_DIR = f"{BASE}/modules"

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)
importlib.invalidate_caches()
from behavior_rules import detect_suspects, DEFAULT_THRESHOLDS

# === 데이터 수집 ===
CARLA_SCENARIOS = ['normal', 'lane_weaving', 'tailgating', 'sudden_lane_change', 'speeding']
CCTV_VIDEOS = ['v01', 'v02', 'v04', 'v05', 'v06']
CONGESTED_VIDEOS = ['v02', 'v06']

TIGHT_TH = {
    'tail_gap': 1.5,
    'lane_change': 5,
    'weaving_std': 0.50,
    'min_track_frames': 60,
}

# CARLA 결과 (기본 임계값)
carla_results = {}
for s in CARLA_SCENARIOS:
    json_path = f"{BASE}/carla_{s}/outputs/test_tracks_v1.1.json"
    carla_results[s] = detect_suspects(json_path)

# CCTV 결과 (극엄격)
cctv_results = {}
for v in CCTV_VIDEOS:
    json_path = f"{BASE}/{v}/outputs/test_tracks_v1.1.json"
    cctv_results[v] = detect_suspects(json_path, thresholds=TIGHT_TH)

# === 보고서 작성 ===
lines = []

lines.append("# Week 4 데모 영상 분석 보고서")
lines.append("")
lines.append(f"**생성일:** {datetime.now().strftime('%Y-%m-%d %H:%M')}")
lines.append(f"**대상:** CARLA 5개 + CCTV 5개 시연 영상")
lines.append(f"**임계값:** CARLA - 기본 / CCTV - 극엄격")
lines.append("")

# 0. 개요
lines.append("## 0. 개요")
lines.append("")
lines.append("본 보고서는 A+C 통합 시스템의 시연 영상 분석 결과이다.")
lines.append("시연 영상은 두 환경에서 생성:")
lines.append("")
lines.append("1. **CARLA 시뮬레이션 (5개)**: 통제 환경, GT 검증 가능")
lines.append("2. **CCTV 실제 환경 (5개)**: 실제 도로, GT 없음, 극엄격 임계값 적용")
lines.append("")

# 1. 시연 영상 목록
lines.append("## 1. 시연 영상 목록")
lines.append("")
lines.append("### 1.1 CARLA 시뮬레이션 영상")
lines.append("")
lines.append("| 파일명 | 시나리오 | 의도된 행동 |")
lines.append("|---|---|---|")
lines.append("| `carla_normal_demo.mp4` | 정상 주행 (대조군) | - |")
lines.append("| `carla_lane_weaving_demo.mp4` | 차선 표류 | lane_weaving |")
lines.append("| `carla_tailgating_demo.mp4` | 짧은 차간거리 | tailgating |")
lines.append("| `carla_sudden_lane_change_demo.mp4` | 급차선 변경 | lane_change |")
lines.append("| `carla_speeding_demo.mp4` | 속도 위반 | speeding |")
lines.append("")
lines.append("**저장 위치:** `driving2_demo/demo_videos/`")
lines.append("")

lines.append("### 1.2 CCTV 영상")
lines.append("")
lines.append("| 파일명 | 위치 | 환경 |")
lines.append("|---|---|---|")
lines.append("| `v01_demo_tight.mp4` | 강서구청 시내 교차로 | 흐름, 3차로 |")
lines.append("| `v02_demo_tight.mp4` | 김포공항 직선 | 정체, 6차로 |")
lines.append("| `v04_demo_tight.mp4` | 성수대교 | 흐름, 4차로 |")
lines.append("| `v05_demo_tight.mp4` | 올림픽대로 | 흐름, 5차로 |")
lines.append("| `v06_demo_tight.mp4` | 양재IC→반포IC | 정체, 4차로 |")
lines.append("")

# 2. CARLA 시연 영상 결과
lines.append("## 2. CARLA 시연 영상 결과")
lines.append("")
lines.append("CARLA 영상은 GT 정답이 있어 룰 정확도 검증 가능.")
lines.append("")

lines.append("### 2.1 시나리오별 결과")
lines.append("")
lines.append("| 시나리오 | 분석 차량 | 의심 차량 | 사유 | 정답 일치 |")
lines.append("|---|---|---|---|---|")
for s in CARLA_SCENARIOS:
    r = carla_results[s]
    n_total = r['total_analyzed']
    n_susp = len(r['suspect_ids'])
    summary_str = ", ".join([f"{k}:{val}" for k, val in r['summary'].items()]) if r['summary'] else "없음"

    if s == 'normal':
        match = "✅ 정확 (대조군, 의심 0대)"
    elif s == 'speeding':
        match = "⚠️ 속도 룰 비활성 (검출 못 함, 예상됨)"
    elif n_susp == 1:
        match = "✅ 정확 (의심 1대)"
    else:
        match = f"⚠️ 불일치"

    lines.append(f"| {s} | {n_total} | {n_susp} | {summary_str} | {match} |")
lines.append("")

lines.append("### 2.2 활성 룰 정확도")
lines.append("")
lines.append("CARLA의 4개 시나리오 중 활성 룰 (lane_weaving, tailgating, lane_change) 검증:")
lines.append("")
lines.append("```")
lines.append("True Positive (TP):  3개")
lines.append("False Positive (FP): 0개 (normal에서 의심 0대)")
lines.append("False Negative (FN): 0개")
lines.append("")
lines.append("Precision = Recall = F1 = 1.00")
lines.append("```")
lines.append("")

lines.append("### 2.3 짚을 점 — Speeding 영상")
lines.append("")
lines.append("**현상:** 속도 위반 영상에서 빠른 차량이 있었으나 시스템에서 검출 못 함.")
lines.append("")
lines.append("**원인:** C 모듈에서 **속도 룰 자체가 비활성**.")
lines.append("- C INTEGRATION_GUIDE: \"속도 위반 룰은 CARLA 속도 분포 한계로 현재 비활성\"")
lines.append("- 임계값 문제가 아니라 룰 자체가 안 돌아감")
lines.append("")
lines.append("**의미:**")
lines.append("- 시스템 결함이 아닌 의도된 설계")
lines.append("- 속도 룰은 향후 활성화 필요 (A 속도 추정 정밀화 + C 룰 검증 데이터)")
lines.append("")

# 3. CCTV 시연 영상 결과
lines.append("## 3. CCTV 시연 영상 결과")
lines.append("")
lines.append("CCTV는 GT 없음 → 절대 정확도 측정 불가, 의심 차량 시각화 분석.")
lines.append("")

lines.append("### 3.1 영상별 결과 (극엄격 임계값)")
lines.append("")
lines.append("**임계값:** tail_gap=1.5m, lane_change=5회, weaving_std=0.50m, min_track_frames=60")
lines.append("")
lines.append("| 영상 | 환경 | 분석 차량 | 의심 차량 | 의심 비율 | 사유 분포 |")
lines.append("|---|---|---|---|---|---|")
for v in CCTV_VIDEOS:
    env = "정체" if v in CONGESTED_VIDEOS else "흐름"
    r = cctv_results[v]
    n_total = r['total_analyzed']
    n_susp = len(r['suspect_ids'])
    ratio = (n_susp / n_total * 100) if n_total > 0 else 0
    summary_str = ", ".join([f"{k}:{val}" for k, val in r['summary'].items()])
    lines.append(f"| {v} | {env} | {n_total} | {n_susp} | {ratio:.1f}% | {summary_str} |")
lines.append("")

lines.append("### 3.2 시각적 관찰")
lines.append("")
lines.append("**v01 (시내 교차로, 흐름):**")
lines.append("- 의심 비율 56% — 절반 이상 빨간 박스")
lines.append("- 주요 사유: tailgating, lane_change")
lines.append("- 한국 시내 도로 특성상 차간거리 짧음")
lines.append("")
lines.append("**v02 (김포공항, 정체):**")
lines.append("- 의심 비율 9% — 가장 합리적")
lines.append("- 분석 차량 적음 (정체로 정적 차량 많아 min_track 60 통과 적음)")
lines.append("")
lines.append("**v04 (성수대교):**")
lines.append("- 의심 비율 86% — 가장 높음")
lines.append("- min_track_frames 60으로 분석 차량 119→36 감소")
lines.append("- 장기 추적 차량은 도시 도로 특성상 tailgating 많이 발생")
lines.append("")
lines.append("**v05 (올림픽대로 5차로):**")
lines.append("- 의심 비율 50%")
lines.append("- lane_change 15건 — 5차로 직선이라 차로 변경 빈번")
lines.append("")
lines.append("**v06 (양재IC, 정체):**")
lines.append("- 의심 비율 50%")
lines.append("- 정체 시 차간거리 짧음 + 차선 표류 자연스러움")
lines.append("")

lines.append("### 3.3 CCTV에서의 성능 저하 원인")
lines.append("")
lines.append("**1. A 시스템 잡음 (호모그래피)**")
lines.append("- YOLO 박스 위치 ±1~2 픽셀 자연 변동")
lines.append("- 호모그래피 변환 시 미터 단위 변동 증폭")
lines.append("- → lateral_offset, lane_id 자연 변동 → false positive")
lines.append("")
lines.append("**2. 한국 도로 특성**")
lines.append("- 차간거리 짧음 (정체 시 1~2m 일상적)")
lines.append("- 다차로 도로 (5차로)에서 차로 변경 빈번")
lines.append("- 운전자 습관 (차선 안에서 미세 흔들림)")
lines.append("")
lines.append("**3. GT 데이터 부재**")
lines.append("- false positive vs true positive 구분 불가")
lines.append("- 의심 비율 50~80%가 진짜인지 잡음인지 판단 어려움")
lines.append("")

# 4. CARLA vs CCTV 비교
lines.append("## 4. CARLA vs CCTV 비교")
lines.append("")
lines.append("| 항목 | CARLA | CCTV |")
lines.append("|---|---|---|")
lines.append("| GT 데이터 | ✅ 있음 | ❌ 없음 |")
lines.append("| 환경 | 통제 시뮬레이션 | 실제 도로 |")
lines.append("| 차량 수 | 5대 | 50~150대 |")
lines.append("| 적용 임계값 | 기본 | 극엄격 |")
lines.append("| 정확도 | 100% (P/R/F1=1.00) | 측정 불가 |")
lines.append("| 의심 비율 | 20% (5대 중 1대) | 평균 50% |")
lines.append("| 시연 효과 | 룰 동작 명확 | 실제 환경 한계 |")
lines.append("")

lines.append("### 4.1 두 영상이 보여주는 것")
lines.append("")
lines.append("**CARLA 시연:** 시스템이 위험 행동을 \"정확히\" 검출 가능함을 입증")
lines.append("- 정상 주행: 의심 0대 (false positive 없음)")
lines.append("- 위험 행동: 정확히 1대 검출 + 사유 일치")
lines.append("")
lines.append("**CCTV 시연:** 실제 환경에서 적용 시 한계 노출")
lines.append("- 한국 도로 특성과 CARLA 기준 임계값의 차이")
lines.append("- A 시스템 잡음의 영향")
lines.append("- GT 부재로 절대 평가 불가")
lines.append("")

# 5. 결론
lines.append("## 5. 결론")
lines.append("")
lines.append("### 5.1 핵심 발견")
lines.append("")
lines.append("1. **CARLA 통제 환경에서 시스템 알고리즘 정확성 입증** (P/R/F1=1.00)")
lines.append("2. **CCTV 실제 환경에서 도메인 격차 (Domain Gap) 확인**")
lines.append("   - CARLA 깔끔한 데이터 vs CCTV 잡음 많은 실제 데이터")
lines.append("   - 임계값 튜닝으로 부분 개선 가능 (의심 비율 72% → 50%)")
lines.append("3. **속도 룰의 의도적 비활성** (향후 활성화 필요)")
lines.append("")

lines.append("### 5.2 시연 영상의 가치")
lines.append("")
lines.append("- **CARLA 시연**: 시스템 \"작동 가능성\" 입증")
lines.append("- **CCTV 시연**: 실제 환경 \"적용 시 고려사항\" 제시")
lines.append("- 두 시연 영상을 함께 보면 시스템의 강점과 한계 모두 명확")
lines.append("")

lines.append("### 5.3 향후 개선 방향")
lines.append("")
lines.append("1. **한국 도로 GT 데이터 확보** → CCTV 정량 평가 가능")
lines.append("2. **A 시스템 시계열 평활화** → lane_id, lateral_offset 잡음 감소")
lines.append("3. **속도 룰 활성화** → 4개 룰 완전 시스템")
lines.append("4. **자동 임계값 학습** → 환경별 동적 조정")
lines.append("")

# 6. 부록 — 시연 영상 시청 가이드
lines.append("## 6. 부록 — 시연 영상 시청 가이드")
lines.append("")
lines.append("### 시각화 요소")
lines.append("")
lines.append("- 🟢 **초록 박스**: 정상 차량")
lines.append("- 🔴 **빨간 박스**: 의심 차량 (위험 행동)")
lines.append("- **자막 (영문)**: 의심 사유")
lines.append("  - `lane_weaving`: 차선 표류")
lines.append("  - `lane_change`: 급차선 변경")
lines.append("  - `tailgating`: 짧은 차간거리")
lines.append("- **상단 정보 바**: 영상 ID, 프레임 번호, 의심 차량 총 수")
lines.append("")

lines.append("### 추천 시청 순서")
lines.append("")
lines.append("1. `carla_normal_demo.mp4` — 정상 주행 (의심 없음)")
lines.append("2. `carla_lane_weaving_demo.mp4` — 차선 표류 검출")
lines.append("3. `carla_tailgating_demo.mp4` — 차간거리 검출")
lines.append("4. `carla_sudden_lane_change_demo.mp4` — 차선 변경 검출")
lines.append("5. `v01_demo_tight.mp4` — 실제 CCTV 적용 사례")
lines.append("6. (선택) 다른 CCTV 영상들")
lines.append("")

# 저장
report_content = "\n".join(lines)
report_path = f"{DEMO_BASE}/week4_demo_report.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"\n✅ Week 4 데모 보고서 저장: {report_path}")
print(f"   총 길이: {len(report_content)} 문자")


✅ Week 4 데모 보고서 저장: /content/drive/MyDrive/driving2_demo/week4_demo_report.md
   총 길이: 4689 문자


In [5]:
# === Week 4 README 작성 ===
from datetime import datetime
import os

DEMO_BASE = "/content/drive/MyDrive/driving2_demo"

readme_lines = []

readme_lines.append("# Week 4 — 시연 영상 및 통합 시스템 데모")
readme_lines.append("")
readme_lines.append(f"**작성일:** {datetime.now().strftime('%Y-%m-%d')}")
readme_lines.append(f"**대상:** A+C 통합 시스템 시연 영상 + 분석 보고서")
readme_lines.append("")

readme_lines.append("## 개요")
readme_lines.append("")
readme_lines.append("3주차에서 완성한 A+C 통합 시스템(영상 처리 + 행동 룰)의 시연 영상을 제작하고 분석한 결과.")
readme_lines.append("")
readme_lines.append("- **CARLA 시뮬레이션 5개**: 통제 환경, GT 검증 가능")
readme_lines.append("- **CCTV 실제 환경 5개**: 실제 도로, 극엄격 임계값 적용")
readme_lines.append("")

readme_lines.append("## 시스템 흐름")
readme_lines.append("")
readme_lines.append("```")
readme_lines.append("영상 (mp4)")
readme_lines.append("    ↓")
readme_lines.append("[A 모듈] YOLOv11 + ByteTrack + 호모그래피")
readme_lines.append("    ↓")
readme_lines.append("JSON v1.1 (차량 위치, 속도, 차로)")
readme_lines.append("    ↓")
readme_lines.append("[C 모듈] 행동 룰 (차선 표류, 차선 변경, 차간거리)")
readme_lines.append("    ↓")
readme_lines.append("의심 차량 시각화 영상 (빨간/초록 박스 + 사유 자막)")
readme_lines.append("```")
readme_lines.append("")

readme_lines.append("## 시연 영상")
readme_lines.append("")
readme_lines.append("**※ 영상 파일은 용량 문제로 GitHub에 업로드되지 않음. 카톡으로 별도 공유.**")
readme_lines.append("")

readme_lines.append("### CARLA 시뮬레이션 영상 (5개)")
readme_lines.append("")
readme_lines.append("기본 임계값 적용. GT 검증으로 룰 정확도 100% 입증.")
readme_lines.append("")
readme_lines.append("| 파일명 | 시나리오 | 의심 차량 | 검출 사유 | 결과 |")
readme_lines.append("|---|---|---|---|---|")
readme_lines.append("| `carla_normal_demo.mp4` | 정상 주행 (대조군) | 0대 | - | ✅ 정확 |")
readme_lines.append("| `carla_lane_weaving_demo.mp4` | 차선 표류 | 1대 | lane_weaving | ✅ 정확 |")
readme_lines.append("| `carla_tailgating_demo.mp4` | 짧은 차간거리 | 1대 | tailgating | ✅ 정확 |")
readme_lines.append("| `carla_sudden_lane_change_demo.mp4` | 급차선 변경 | 1대 | lane_change | ✅ 정확 |")
readme_lines.append("| `carla_speeding_demo.mp4` | 속도 위반 | 0대 | (룰 비활성) | ⚠️ 검출 안됨 |")
readme_lines.append("")

readme_lines.append("### CCTV 실제 영상 (5개)")
readme_lines.append("")
readme_lines.append("극엄격 임계값 적용. GT 없어 절대 정확도 측정 불가.")
readme_lines.append("")
readme_lines.append("**임계값:** tail_gap=1.5m, lane_change=5회, weaving_std=0.50m, min_track_frames=60")
readme_lines.append("")
readme_lines.append("| 파일명 | 위치 | 환경 | 의심 비율 |")
readme_lines.append("|---|---|---|---|")
readme_lines.append("| `v01_demo_tight.mp4` | 강서구청 시내 교차로 | 흐름, 3차로 | 56.2% |")
readme_lines.append("| `v02_demo_tight.mp4` | 김포공항 직선 | 정체, 6차로 | 9.1% |")
readme_lines.append("| `v04_demo_tight.mp4` | 성수대교 | 흐름, 4차로 | 86.1% |")
readme_lines.append("| `v05_demo_tight.mp4` | 올림픽대로 | 흐름, 5차로 | 50.0% |")
readme_lines.append("| `v06_demo_tight.mp4` | 양재IC→반포IC | 정체, 4차로 | 50.0% |")
readme_lines.append("")

readme_lines.append("## 시연 영상 시청 가이드")
readme_lines.append("")
readme_lines.append("### 시각화 요소")
readme_lines.append("")
readme_lines.append("- 🟢 **초록 박스**: 정상 차량")
readme_lines.append("- 🔴 **빨간 박스**: 의심 차량 (위험 행동)")
readme_lines.append("- **자막 (영문)**: 의심 사유")
readme_lines.append("  - `lane_weaving`: 차선 표류")
readme_lines.append("  - `lane_change`: 급차선 변경")
readme_lines.append("  - `tailgating`: 짧은 차간거리")
readme_lines.append("- **상단 정보 바**: 영상 ID, 프레임 번호, 의심 차량 총 수")
readme_lines.append("")

readme_lines.append("### 추천 시청 순서")
readme_lines.append("")
readme_lines.append("1. `carla_normal_demo.mp4` — 정상 주행 (의심 없음, 대조군)")
readme_lines.append("2. `carla_lane_weaving_demo.mp4` — 차선 표류 검출")
readme_lines.append("3. `carla_tailgating_demo.mp4` — 차간거리 검출")
readme_lines.append("4. `carla_sudden_lane_change_demo.mp4` — 차선 변경 검출")
readme_lines.append("5. `v01_demo_tight.mp4` — 실제 CCTV 적용 사례")
readme_lines.append("6. (선택) 다른 CCTV 영상들")
readme_lines.append("")

readme_lines.append("## 주요 결과")
readme_lines.append("")

readme_lines.append("### ✅ CARLA 통제 환경 — 알고리즘 정확성 입증")
readme_lines.append("")
readme_lines.append("- 활성 룰 3개 정확도: **Precision = Recall = F1 = 1.00**")
readme_lines.append("- False positive 0% (normal 영상에서 의심 0대)")
readme_lines.append("- 사유 매칭 100% (lane_weaving, tailgating, lane_change 모두 정확)")
readme_lines.append("")

readme_lines.append("### ⚠️ CCTV 실제 환경 — 도메인 격차 발견")
readme_lines.append("")
readme_lines.append("- 극엄격 임계값 적용에도 의심 비율 평균 50%")
readme_lines.append("- 한국 도시 도로 특성 (짧은 차간거리, 잦은 차로 변경)")
readme_lines.append("- A 시스템 잡음 (호모그래피, lane_id 변동) 영향")
readme_lines.append("- GT 부재로 false positive vs true positive 구분 불가")
readme_lines.append("")

readme_lines.append("### ⚠️ Speeding 룰 비활성")
readme_lines.append("")
readme_lines.append("- C 모듈에서 속도 위반 룰 의도적 비활성")
readme_lines.append("- 이유: CARLA 속도 분포 한계로 검증 어려움")
readme_lines.append("- 향후 A 속도 추정 정밀화 + 검증 데이터 확보 시 활성화 예정")
readme_lines.append("")

readme_lines.append("## 보고서")
readme_lines.append("")
readme_lines.append("- **`week4_demo_report.md`**: 시연 영상 상세 분석")
readme_lines.append("  - CARLA 시연 결과")
readme_lines.append("  - CCTV 시연 결과")
readme_lines.append("  - CARLA vs CCTV 비교")
readme_lines.append("  - 성능 저하 원인 분석")
readme_lines.append("  - 향후 개선 방향")
readme_lines.append("")
readme_lines.append("## 이전 단계와의 관계")
readme_lines.append("")
readme_lines.append("- **week1** (`../week1/`): 기본 YOLO+ByteTrack 파이프라인 구축")
readme_lines.append("- **week2** (`../week2/`): 호모그래피 캘리브레이션 도입, schema v1.1")
readme_lines.append("- **week3** (`../week3/`): 다중 영상 처리 + CARLA GT + C 모듈 통합")
readme_lines.append("- **week4** (본 폴더): 시연 영상 제작 + 종합 분석")
readme_lines.append("")
readme_lines.append("**week3의 통합 보고서** (`../week3/ac_integration_report.md`)에 시스템 검증 결과 상세 기록.")
readme_lines.append("**week4의 데모 보고서** (`week4_demo_report.md`)는 시연 영상 분석에 집중.")
readme_lines.append("")

readme_lines.append("## 폴더 구성")
readme_lines.append("")
readme_lines.append("```")
readme_lines.append("A/week4/")
readme_lines.append("├── README.md                # 본 문서")
readme_lines.append("└── week4_demo_report.md     # 시연 영상 분석 보고서")
readme_lines.append("```")
readme_lines.append("")
readme_lines.append("**시연 영상 파일 (별도 공유):**")
readme_lines.append("- CARLA 5개: `carla_*_demo.mp4`")
readme_lines.append("- CCTV 5개: `v*_demo_tight.mp4`")
readme_lines.append("")

readme_lines.append("## 시스템 강점과 한계 요약")
readme_lines.append("")

readme_lines.append("### 강점")
readme_lines.append("")
readme_lines.append("- ✅ CARLA 통제 환경에서 100% 정확도 달성")
readme_lines.append("- ✅ A → C 데이터 인터페이스 안정성 (JSON v1.1)")
readme_lines.append("- ✅ 임계값 동적 조정으로 환경 적응 가능")
readme_lines.append("")

readme_lines.append("### 한계")
readme_lines.append("")
readme_lines.append("- ⚠️ CCTV 실제 환경 정량 평가 어려움 (GT 부재)")
readme_lines.append("- ⚠️ A 시스템 잡음으로 false positive 발생 가능")
readme_lines.append("- ⚠️ 속도 룰 비활성 (활성화 필요)")
readme_lines.append("- ⚠️ 한국 도로 특성과 CARLA 기준 임계값 간 격차")
readme_lines.append("")

readme_lines.append("### 향후 개선")
readme_lines.append("")
readme_lines.append("1. 한국 도로 GT 데이터 확보 → CCTV 정량 평가")
readme_lines.append("2. A 시계열 평활화 (Kalman Filter) → 잡음 감소")
readme_lines.append("3. 속도 룰 활성화 → 4개 룰 완전 시스템")
readme_lines.append("4. 자동 임계값 학습 → 환경별 동적 조정")
readme_lines.append("")

readme_lines.append("## 팀 구성")
readme_lines.append("")
readme_lines.append("- **A 모듈**: 영상 처리 파이프라인 (본 작업)")
readme_lines.append("- **B 모듈**: CARLA 시뮬레이션 + GT 생성")
readme_lines.append("- **C 모듈**: 행동 룰 + 위험 차량 판단")
readme_lines.append("")

# 저장
readme_content = "\n".join(readme_lines)
readme_path = f"{DEMO_BASE}/week4_README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"✅ Week 4 README 저장: {readme_path}")
print(f"   총 길이: {len(readme_content)} 문자")

✅ Week 4 README 저장: /content/drive/MyDrive/driving2_demo/week4_README.md
   총 길이: 3488 문자
